加州房价预测的步骤可以分为以下几个核心环节，用文字描述如下：
1. 数据加载与初步探索
加载加州房价数据集（比如从文件读取 CSV 格式数据）；
查看数据基本信息：比如前几行数据、数据形状（行数 / 列数）、统计特征（均值、最值等）；
分析数据分布：比如用直方图看关键特征（如收入中位数）的分布，查看类别型特征（如地理位置）的取值情况。
2. 数据预处理
分离特征与标签：把数据集拆分为 “特征（X，比如经纬度、收入等）” 和 “标签（y，即房价中位数）”；
数据拆分：用train_test_split将特征和标签分别拆分为训练集（X_train、y_train）和测试集（X_test、y_test），用于后续训练和评估；
（可选）处理缺失值、类别特征编码、特征缩放等（比如用SimpleImputer补缺失值，StandardScaler做标准化）。
3. 模型训练
选择模型：比如用线性回归（LinearRegression）作为基础模型；
训练模型：用训练集的特征（X_train）和标签（y_train）拟合模型。
4. 模型评估
用训练好的模型预测测试集特征（X_test），得到预测结果；
用评估指标（比如均方误差 MSE、R² 分数）对比预测结果与测试集真实标签（y_test），判断模型效果。
5. （可选）模型优化
比如尝试其他模型（决策树、随机森林），或调整模型参数，进一步提升预测精度。

第一步，导入基本库

In [ ]:
from sklearn.model_selection import train_test_split   #划分训练集，测试集
from sklearn.preprocessing import StandardScaler #标准化
from sklearn.linear_model import LinearRegression  #线性回归 
from sklearn.impute import SimpleImputer #补充缺失值
from sklearn.metrics import mean_squared_error, r2_score #预测与评估
import pandas as pd  #读取数据
import numpy as np  #数组


第二步，读取数据

使用pandas读取数据

In [ ]:
housing=pd.read_csv('F:/Project/work/words/scikitLearn/datasets/housing/housing.csv') 
print(housing.info()) 

读取数据后如下

Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object

可以看出total_bedrooms处有缺失值，因此在后面需要进行补充，先分类出特征与标签

In [ ]:
X = housing.drop("median_house_value", axis=1)  # 特征：去掉标签列
y = housing["median_house_value"]  # 标签：目标列 

由于存在float与object两种数据类型，所以要将其分开处理，float类型使用标准化进行处理，object类型使用OneHot编码进行处理

分离数值特征与分类特征

In [ ]:
#分离数值特征和分类特征
num_features = X.select_dtypes(include=['int64', 'float64']).columns  # 自动识别数值列
cat_features = ['ocean_proximity']  # 分类列

然后划分为训练集与测试集（一般比例为8:2）

In [ ]:
#划分训练集，测试集
X_train,X_test,y_train,y_test = train_test_split( 
    X,y,test_size=0.2,random_state=42 
)


先对float类型的数据进行标准化

标准化处理的一个关键好处是它不受特征原始尺度的影响，
因为转换是基于每个特征自身的统计属性（均值和标准差）进行的。
这意味着无论特征的原始值是大是小，标准化后都将具有相同的尺度，
从而有助于许多机器学习算法的性能提升。

In [ ]:
# 标准化数值特征（仅针对数值列）
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)  # 训练集：拟合+转换
X_test_num_scaled = scaler.transform(X_test_num)        # 测试集：仅转换（关键！）

对object数据使用OneHot编码

In [ ]:
# 对分类特征做OneHot编码
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')  
X_train_cat = encoder.fit_transform(X_train[cat_features])
X_test_cat = encoder.transform(X_test[cat_features])  


将处理好的数据整合到一起

In [ ]:
#把标准化后的数值特征 + 独热编码后的分类特征合并
X_train_processed = np.hstack([X_train_num_scaled, X_train_cat])
X_test_processed = np.hstack([X_test_num_scaled, X_test_cat])


In [ ]:
#把标准化后的数值特征 + 独热编码后的分类特征合并
X_train_processed = np.hstack([X_train_num_scaled, X_train_cat])
X_test_processed = np.hstack([X_test_num_scaled, X_test_cat])

现在可以使用线性回归进行拟合

In [ ]:
#使用线性回归模型进行拟合 
linear = LinearRegression()
linear.fit(X_train_processed, y_train)

最后进行预测与评估

In [ ]:
#预测
y_pred = linear.predict(X_test_processed)
#评估
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

输出结果

In [ ]:
print(f"均方误差: {mse:.4f}")
print(f"决定系数: {r2:.4f}")

完整代码

In [ ]:
import pandas as pd 
from sklearn.model_selection import train_test_split    
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
housing=pd.read_csv('F:/Project/work/words/scikitLearn/datasets/housing/housing.csv') 
 
 # 先分离特征和标签
X = housing.drop("median_house_value", axis=1)  # 特征：去掉标签列
y = housing["median_house_value"]  # 标签：目标列  


#分离数值特征和分类特征
num_features = X.select_dtypes(include=['int64', 'float64']).columns  # 自动识别数值列
cat_features = ['ocean_proximity']  # 分类列

#划分训练集，测试集
X_train,X_test,y_train,y_test = train_test_split( 
    X,y,test_size=0.2,random_state=42
) 

# 填充数值特征的空值（避免标准化时报错）
num_imputer = SimpleImputer(strategy='median')  # 用中位数填充更稳健
X_train_num = num_imputer.fit_transform(X_train[num_features])
X_test_num = num_imputer.transform(X_test[num_features])  # 测试集复用训练集的填充规则

# 标准化数值特征（仅针对数值列）
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)  # 训练集：拟合+转换
X_test_num_scaled = scaler.transform(X_test_num)        # 测试集：仅转换（关键！）


# 对分类特征做OneHot编码
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')  # 非稀疏矩阵更易理解
X_train_cat = encoder.fit_transform(X_train[cat_features])
X_test_cat = encoder.transform(X_test[cat_features])  # 测试集复用训练集的编码规则


#把标准化后的数值特征 + 独热编码后的分类特征合并
X_train_processed = np.hstack([X_train_num_scaled, X_train_cat])
X_test_processed = np.hstack([X_test_num_scaled, X_test_cat])

# 训练线性回归模型
linear = LinearRegression()
linear.fit(X_train_processed, y_train)

#预测
y_pred = linear.predict(X_test_processed)

#评估
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"均方误差: {mse:.4f}")
print(f"决定系数: {r2:.4f}")